In [ ]:
import pandas as pd
import psycopg2
import json
from psycopg2 import extras

In [ ]:
dsn = "dbname=training user=postgres password=kenan123 host=localhost port=5432"
q = """SELECT * FROM raw_products"""
with psycopg2.connect(dsn) as con:
        with con.cursor() as cur:
            cur.execute(q)
            result = cur.fetchall()
        con.commit()



In [ ]:
df = pd.DataFrame(result , columns=['id' , 'title' , 'category' , 'price' , 'discount_percentage' , 'rating' , 'stock' , 'brand' , 'sku' , 'extracted_at'])

In [ ]:
df.info()

### After seeing the data we shoud first change the dtype then fix the null values

In [ ]:
df['price'] = df['price'].astype(float)
df['discount_percentage'] = df['discount_percentage'].astype(float)
df['rating'] = df['rating'].astype(float)

In [ ]:
df['brand'] = df['brand'].fillna('UNKNOWN_BRAND')

In [ ]:
df['category'].unique()

In [ ]:
df['category'] = df['category'].replace({
    "motorcycle" : "motor-cycle",
    "smartphones" : "smart-phones"
})

In [ ]:
df.info()

In [ ]:
mask = (df['rating'] < 0).sum()
print(mask)

In [ ]:
mask = (df['price'] < 0).sum()
print(mask)

In [ ]:
mask = (df['discount_percentage'] < 0).sum()
print(mask)

In [ ]:
df.info()

In [ ]:
df['priceAfterDiscount'] = (df['price'] * (1 - (df['discount_percentage'].fillna(0) / 100))).round(2)
cols = ['id' , 'title' , 'category' , 'price' , 'discount_percentage' ,'priceAfterDiscount','rating' , 'stock' , 'brand' , 'sku' , 'extracted_at']
df = df[cols]

In [ ]:
df.head()

In [ ]:
q = """
    INSERT INTO cleaned_raw_products (id, title, category, price, discountPercentage,priceAfterDiscount, rating, stock, brand, sku,extracted_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s,%s,%s)
    ON CONFLICT (id) DO NOTHING;
    """
data = [tuple(x) for x in df.values]
with psycopg2.connect(dsn) as con:
        with con.cursor() as cur:
            extras.execute_batch(cur,q,data)
        con.commit()